# Open Street Map API Test - Improved Version

This notebook queries OpenStreetMap for amenities and calculates walkability scores.

**Key Improvements:**
- Added shop=supermarket and shop=convenience for groceries
- Added leisure=park for parks
- Added highway=bus_stop for transit
- Handle both nodes and ways (with center coordinates)
- Improved walkability scoring algorithm

In [1]:
import requests
from math import radians, cos, sin, asin, sqrt
import json

## 1. Query OSM API (Improved)

In [2]:
def query_osm_amenities(lat, lon, radius_meters=1600):
  """
  Comprehensive OSM query for all relevant amenities.
  
  Queries for:
  - amenity tags (restaurants, schools, hospitals, etc.)
  - shop tags (supermarkets, convenience stores)
  - leisure tags (parks, gardens)
  - highway tags (bus stops)
  
  Returns both nodes and ways with center coordinates.
  """
  overpass_url = "http://overpass-api.de/api/interpreter"
  
  query = f"""
  [out:json][timeout:25];
  (
  node["amenity"](around:{radius_meters},{lat},{lon});
  way["amenity"](around:{radius_meters},{lat},{lon});
  node["shop"="supermarket"](around:{radius_meters},{lat},{lon});
  way["shop"="supermarket"](around:{radius_meters},{lat},{lon});
  node["shop"="convenience"](around:{radius_meters},{lat},{lon});
  way["shop"="convenience"](around:{radius_meters},{lat},{lon});
  node["leisure"="park"](around:{radius_meters},{lat},{lon});
  way["leisure"="park"](around:{radius_meters},{lat},{lon});
  node["highway"="bus_stop"](around:{radius_meters},{lat},{lon});
  );
  out center body;
  """
  
  response = requests.post(overpass_url, data={"data": query})
  return response.json()

## 2. Test with Orlando Coordinates

In [3]:
# TEST with Orlando downtown coordinates
test_data = query_osm_amenities(28.5383, -81.3792)
print(f"Retrieved {len(test_data['elements'])} elements from OSM")

Retrieved 614 elements from OSM


In [4]:
# EXPLORE the structure
print("Response keys:", test_data.keys())
print("\nFirst element:")
print(json.dumps(test_data['elements'][0], indent=2))

Response keys: dict_keys(['version', 'generator', 'osm3s', 'elements'])

First element:
{
  "type": "node",
  "id": 358693547,
  "lat": 28.5358355,
  "lon": -81.3692363,
  "tags": {
    "amenity": "school",
    "ele": "26",
    "gnis:feature_id": "280355",
    "name": "Cherokee School",
    "operator": "Orange County Public Schools",
    "operator:short": "OCPS",
    "operator:type": "public",
    "operator:wikidata": "Q8778313"
  }
}


## 3. Analyze What Amenities We Found

In [5]:
def analyze_amenities(data):
    """
    Analyze all tags (amenity, shop, leisure, highway) to see what we got.
    """
    tag_types = {}
    
    for element in data['elements']:
        tags = element.get('tags', {})
        
        # Check all relevant tag types
        for tag_key in ['amenity', 'shop', 'leisure', 'highway']:
            if tag_key in tags:
                full_tag = f"{tag_key}={tags[tag_key]}"
                tag_types[full_tag] = tag_types.get(full_tag, 0) + 1
    
    return tag_types

# Run analysis
result = query_osm_amenities(28.5383, -81.3792)
tag_counts = analyze_amenities(result)

print("All tags found (sorted by count):")
print("="*50)
for tag, count in sorted(tag_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {tag}: {count}")

All tags found (sorted by count):
  highway=bus_stop: 171
  amenity=parking: 134
  amenity=parking_entrance: 39
  amenity=restaurant: 39
  amenity=bicycle_parking: 38
  amenity=place_of_worship: 24
  amenity=fountain: 22
  amenity=cafe: 21
  amenity=bar: 19
  amenity=school: 13
  amenity=bank: 12
  leisure=park: 11
  amenity=fast_food: 9
  amenity=clinic: 7
  shop=convenience: 4
  amenity=nightclub: 4
  amenity=shelter: 4
  amenity=drinking_water: 4
  amenity=theatre: 3
  amenity=hospital: 3
  amenity=courthouse: 3
  amenity=arts_centre: 2
  shop=supermarket: 2
  amenity=doctors: 2
  amenity=toilets: 2
  amenity=pharmacy: 2
  amenity=post_office: 2
  shop=copyshop: 2
  amenity=community_centre: 2
  amenity=fuel: 1
  amenity=bicycle_rental: 1
  amenity=cinema: 1
  amenity=studio: 1
  amenity=events_venue: 1
  amenity=watering_place: 1
  amenity=ice_cream: 1
  amenity=atm: 1
  amenity=dentist: 1
  amenity=parking_space: 1
  amenity=library: 1
  amenity=college: 1
  amenity=public_buildin

## 4. Distance Calculation

In [6]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two points in miles using Haversine formula.
    """
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    
    # Radius of earth in miles
    miles = 3956 * c
    return miles

def get_element_location(element):
    """
    Get lat/lon for both nodes and ways.
    For ways, use the 'center' coordinates provided by OSM.
    """
    if element.get('type') == 'node':
        return element.get('lat'), element.get('lon')
    elif element.get('type') == 'way':
        # For ways, OSM includes center when we use 'out center'
        if 'center' in element:
            return element['center']['lat'], element['center']['lon']
        return None, None
    return None, None

# Test distance calculation
property_lat, property_lon = 28.5383, -81.3792
test_element = test_data['elements'][0]
elem_lat, elem_lon = get_element_location(test_element)

if elem_lat and elem_lon:
    distance = haversine_distance(property_lat, property_lon, elem_lat, elem_lon)
    name = test_element.get('tags', {}).get('name', 'Unknown')
    print(f"Distance to {name}: {distance:.2f} miles")

Distance to Cherokee School: 0.63 miles


## 5. Process Amenities into Structured Data

In [7]:
def process_amenities(raw_data, property_lat, property_lon):
    """
    Transform OSM data into structured insights.
    Now handles amenity, shop, leisure, and highway tags.
    """
    elements = raw_data.get('elements', [])
    
    # Categorize amenities
    groceries = []
    restaurants = []
    schools = []
    parks = []
    hospitals = []
    transit = []
    cafes = []
    
    for element in elements:
        tags = element.get('tags', {})
        
        # Get all relevant tag types
        amenity_type = tags.get('amenity')
        shop_type = tags.get('shop')
        leisure_type = tags.get('leisure')
        highway_type = tags.get('highway')
        
        # Get location (works for both nodes and ways)
        elem_lat, elem_lon = get_element_location(element)
        
        if not (elem_lat and elem_lon):
            continue
        
        # Calculate distance
        distance = haversine_distance(property_lat, property_lon, elem_lat, elem_lon)
        
        item = {
            'name': tags.get('name', tags.get('operator', 'Unknown')),
            'distance_miles': round(distance, 2),
            'coordinates': {'lat': elem_lat, 'lon': elem_lon}
        }
        
        # Categorize - now checking multiple tag types
        if shop_type in ['supermarket', 'convenience'] or amenity_type in ['supermarket', 'grocery']:
            item['type'] = shop_type or amenity_type
            groceries.append(item)
        
        elif amenity_type == 'restaurant':
            restaurants.append(item)
        
        elif amenity_type == 'cafe':
            cafes.append(item)
        
        elif amenity_type == 'school':
            schools.append(item)
        
        elif leisure_type == 'park' or amenity_type in ['park', 'garden']:
            parks.append(item)
        
        elif amenity_type in ['hospital', 'clinic', 'doctors']:
            item['type'] = amenity_type
            hospitals.append(item)
        
        elif highway_type == 'bus_stop' or amenity_type in ['bus_station', 'bus_stop']:
            transit.append(item)
    
    # Sort all by distance
    for category in [groceries, restaurants, schools, parks, hospitals, transit, cafes]:
        category.sort(key=lambda x: x['distance_miles'])
    
    # Build structured output
    return {
        'property_location': {
            'lat': property_lat,
            'lon': property_lon
        },
        'groceries': {
            'count_within_1mile': len([g for g in groceries if g['distance_miles'] <= 1.0]),
            'nearest': groceries[0] if groceries else None,
            'all': groceries[:5]
        },
        'restaurants': {
            'count_within_1mile': len([r for r in restaurants if r['distance_miles'] <= 1.0]),
            'nearest': restaurants[0] if restaurants else None,
            'total_found': len(restaurants)
        },
        'cafes': {
            'count_within_1mile': len([c for c in cafes if c['distance_miles'] <= 1.0]),
            'nearest': cafes[0] if cafes else None,
        },
        'schools': {
            'count_within_1mile': len([s for s in schools if s['distance_miles'] <= 1.0]),
            'nearest': schools[0] if schools else None,
            'all': schools[:3]
        },
        'parks': {
            'count_within_1mile': len([p for p in parks if p['distance_miles'] <= 1.0]),
            'nearest': parks[0] if parks else None,
            'all': parks[:3]
        },
        'hospitals': {
            'nearest': hospitals[0] if hospitals else None,
            'count_within_5miles': len([h for h in hospitals if h['distance_miles'] <= 5.0]),
            'all': hospitals[:3]
        },
        'transit': {
            'nearest_stop': transit[0] if transit else None,
            'count_within_half_mile': len([t for t in transit if t['distance_miles'] <= 0.5]),
            'count_within_1mile': len([t for t in transit if t['distance_miles'] <= 1.0])
        }
    }

# Test processing
raw_data = query_osm_amenities(28.5383, -81.3792)
processed = process_amenities(raw_data, 28.5383, -81.3792)

print("\nProcessed Amenities Summary:")
print("="*50)
print(json.dumps(processed, indent=2))


Processed Amenities Summary:
{
  "property_location": {
    "lat": 28.5383,
    "lon": -81.3792
  },
  "groceries": {
    "count_within_1mile": 6,
    "nearest": {
      "name": "Plaza Market & Spirits",
      "distance_miles": 0.19,
      "coordinates": {
        "lat": 28.541059,
        "lon": -81.3792529
      },
      "type": "convenience"
    },
    "all": [
      {
        "name": "Plaza Market & Spirits",
        "distance_miles": 0.19,
        "coordinates": {
          "lat": 28.541059,
          "lon": -81.3792529
        },
        "type": "convenience"
      },
      {
        "name": "Indoor Farmer Market",
        "distance_miles": 0.26,
        "coordinates": {
          "lat": 28.5417616,
          "lon": -81.3776219
        },
        "type": "supermarket"
      },
      {
        "name": "Jr Food Store",
        "distance_miles": 0.36,
        "coordinates": {
          "lat": 28.5385258,
          "lon": -81.3850895
        },
        "type": "convenience"
      },

## 6. Calculate Walkability Score (Improved Algorithm)

In [8]:
def calculate_walkability_score(amenities_data):
    """
    Improved walkability calculation (0-100).
    Based on proximity and variety of amenities.
    
    Scoring breakdown:
    - Groceries: 25 points max
    - Restaurants/Dining: 25 points max
    - Parks/Recreation: 20 points max
    - Transit: 15 points max
    - Schools: 15 points max
    """
    score = 0
    
    # Grocery accessibility (max 25 points)
    grocery_count = amenities_data['groceries']['count_within_1mile']
    nearest_grocery = amenities_data['groceries']['nearest']
    
    if nearest_grocery:
        distance = nearest_grocery['distance_miles']
        if distance <= 0.25:
            score += 25  # Very walkable
        elif distance <= 0.5:
            score += 20  # Quite walkable
        elif distance <= 0.75:
            score += 15  # Moderately walkable
        elif distance <= 1.0:
            score += 10  # Barely walkable
        else:
            score += 5   # Not really walkable
    
    # Bonus for multiple grocery options
    if grocery_count >= 3:
        score += 5
    
    # Restaurant/dining variety (max 25 points)
    restaurant_count = amenities_data['restaurants']['count_within_1mile']
    cafe_count = amenities_data['cafes']['count_within_1mile']
    dining_total = restaurant_count + cafe_count
    
    if dining_total >= 30:
        score += 25  # Excellent dining scene
    elif dining_total >= 20:
        score += 20  # Great dining options
    elif dining_total >= 10:
        score += 15  # Good variety
    elif dining_total >= 5:
        score += 10  # Some options
    else:
        score += dining_total * 2  # Minimal options
    
    # Parks/recreation (max 20 points)
    park_count = amenities_data['parks']['count_within_1mile']
    nearest_park = amenities_data['parks']['nearest']
    
    if nearest_park:
        distance = nearest_park['distance_miles']
        if distance <= 0.25:
            score += 15  # Park very close
        elif distance <= 0.5:
            score += 12  # Park nearby
        elif distance <= 1.0:
            score += 8   # Park accessible
    
    # Bonus for multiple parks
    score += min(5, park_count * 2)
    
    # Transit access (max 15 points)
    transit_half_mile = amenities_data['transit']['count_within_half_mile']
    transit_one_mile = amenities_data['transit']['count_within_1mile']
    
    if transit_half_mile >= 5:
        score += 15  # Excellent transit
    elif transit_half_mile >= 3:
        score += 12  # Good transit
    elif transit_half_mile >= 1:
        score += 8   # Some transit
    elif transit_one_mile >= 3:
        score += 5   # Transit accessible but not close
    
    # Schools (max 15 points)
    school_count = amenities_data['schools']['count_within_1mile']
    nearest_school = amenities_data['schools']['nearest']
    
    if nearest_school:
        distance = nearest_school['distance_miles']
        if distance <= 0.5:
            score += 10  # School very close
        elif distance <= 1.0:
            score += 7   # School nearby
    
    # Bonus for multiple schools
    if school_count >= 5:
        score += 5
    elif school_count >= 3:
        score += 3
    
    return min(100, score)  # Cap at 100

# Calculate walkability
processed = process_amenities(raw_data, 28.5383, -81.3792)
walkability = calculate_walkability_score(processed)

print(f"\n{'='*50}")
print(f"Walkability Score: {walkability}/100")
print(f"{'='*50}")

# Interpret the score
if walkability >= 90:
    interpretation = "Walker's Paradise - Daily errands do not require a car"
elif walkability >= 70:
    interpretation = "Very Walkable - Most errands can be accomplished on foot"
elif walkability >= 50:
    interpretation = "Somewhat Walkable - Some errands can be accomplished on foot"
elif walkability >= 25:
    interpretation = "Car-Dependent - Most errands require a car"
else:
    interpretation = "Very Car-Dependent - Almost all errands require a car"

print(f"\nInterpretation: {interpretation}")


Walkability Score: 100/100

Interpretation: Walker's Paradise - Daily errands do not require a car


## 7. Test Multiple Orlando Locations

In [9]:
# Test different Orlando neighborhoods
test_locations = [
    {"name": "Downtown Orlando", "lat": 28.5383, "lon": -81.3792},
    {"name": "Winter Park", "lat": 28.5999, "lon": -81.3393},
    {"name": "Lake Nona", "lat": 28.3852, "lon": -81.2637},
    {"name": "College Park", "lat": 28.5675, "lon": -81.3992}
]

print("\n" + "="*70)
print("COMPARING ORLANDO NEIGHBORHOODS")
print("="*70)

results = []

for location in test_locations:
    print(f"\n{'='*70}")
    print(f"Location: {location['name']}")
    print(f"Coordinates: ({location['lat']}, {location['lon']})")
    print('='*70)
    
    try:
        raw = query_osm_amenities(location['lat'], location['lon'])
        processed = process_amenities(raw, location['lat'], location['lon'])
        score = calculate_walkability_score(processed)
        
        print(f"\nWalkability Score: {score}/100")
        print(f"\nAmenities Summary:")
        print(f"  Groceries within 1mi: {processed['groceries']['count_within_1mile']}")
        print(f"  Restaurants within 1mi: {processed['restaurants']['count_within_1mile']}")
        print(f"  Cafes within 1mi: {processed['cafes']['count_within_1mile']}")
        print(f"  Parks within 1mi: {processed['parks']['count_within_1mile']}")
        print(f"  Schools within 1mi: {processed['schools']['count_within_1mile']}")
        print(f"  Transit stops within 0.5mi: {processed['transit']['count_within_half_mile']}")
        
        if processed['groceries']['nearest']:
            print(f"\n  Nearest grocery: {processed['groceries']['nearest']['name']} ({processed['groceries']['nearest']['distance_miles']} mi)")
        if processed['parks']['nearest']:
            print(f"  Nearest park: {processed['parks']['nearest']['name']} ({processed['parks']['nearest']['distance_miles']} mi)")
        
        results.append({
            'name': location['name'],
            'score': score,
            'data': processed
        })
        
    except Exception as e:
        print(f"Error processing {location['name']}: {e}")

# Summary comparison
print(f"\n\n{'='*70}")
print("WALKABILITY RANKING")
print("="*70)

results.sort(key=lambda x: x['score'], reverse=True)
for i, result in enumerate(results, 1):
    print(f"{i}. {result['name']}: {result['score']}/100")


COMPARING ORLANDO NEIGHBORHOODS

Location: Downtown Orlando
Coordinates: (28.5383, -81.3792)

Walkability Score: 100/100

Amenities Summary:
  Groceries within 1mi: 6
  Restaurants within 1mi: 39
  Cafes within 1mi: 21
  Parks within 1mi: 10
  Schools within 1mi: 12
  Transit stops within 0.5mi: 61

  Nearest grocery: Plaza Market & Spirits (0.19 mi)
  Nearest park: Heritage Square Park (0.3 mi)

Location: Winter Park
Coordinates: (28.5999, -81.3393)

Walkability Score: 75/100

Amenities Summary:
  Groceries within 1mi: 3
  Restaurants within 1mi: 21
  Cafes within 1mi: 6
  Parks within 1mi: 7
  Schools within 1mi: 0
  Transit stops within 0.5mi: 9

  Nearest grocery: Unknown (0.71 mi)
  Nearest park: Alberta/Cortland Mini Park (0.2 mi)

Location: Lake Nona
Coordinates: (28.3852, -81.2637)

Walkability Score: 36/100

Amenities Summary:
  Groceries within 1mi: 1
  Restaurants within 1mi: 1
  Cafes within 1mi: 0
  Parks within 1mi: 7
  Schools within 1mi: 1
  Transit stops within 0.5mi:

## 8. Export Function for MCP Server

This is the final function ready to be copied into MCP server.

In [10]:
def get_property_amenities_and_walkability(latitude: float, longitude: float, radius_miles: float = 1.0) -> dict:
    """
    Complete function ready for MCP server.
    
    Args:
        latitude: Property latitude
        longitude: Property longitude
        radius_miles: Search radius in miles (default 1.0)
    
    Returns:
        Dict containing processed amenities and walkability score
    """
    radius_meters = int(radius_miles * 1609.34)
    
    # Query OSM
    raw_data = query_osm_amenities(latitude, longitude, radius_meters)
    
    # Process amenities
    processed = process_amenities(raw_data, latitude, longitude)
    
    # Calculate walkability
    walkability_score = calculate_walkability_score(processed)
    
    # Add walkability to output
    processed['walkability_score'] = walkability_score
    
    # Add interpretation
    if walkability_score >= 90:
        interpretation = "Walker's Paradise"
    elif walkability_score >= 70:
        interpretation = "Very Walkable"
    elif walkability_score >= 50:
        interpretation = "Somewhat Walkable"
    elif walkability_score >= 25:
        interpretation = "Car-Dependent"
    else:
        interpretation = "Very Car-Dependent"
    
    processed['walkability_interpretation'] = interpretation
    
    return processed

# Test the export function
test_result = get_property_amenities_and_walkability(28.5383, -81.3792)
print("\n" + "="*70)
print("FINAL OUTPUT (Ready for MCP Server)")
print("="*70)
print(json.dumps(test_result, indent=2))


FINAL OUTPUT (Ready for MCP Server)
{
  "property_location": {
    "lat": 28.5383,
    "lon": -81.3792
  },
  "groceries": {
    "count_within_1mile": 6,
    "nearest": {
      "name": "Plaza Market & Spirits",
      "distance_miles": 0.19,
      "coordinates": {
        "lat": 28.541059,
        "lon": -81.3792529
      },
      "type": "convenience"
    },
    "all": [
      {
        "name": "Plaza Market & Spirits",
        "distance_miles": 0.19,
        "coordinates": {
          "lat": 28.541059,
          "lon": -81.3792529
        },
        "type": "convenience"
      },
      {
        "name": "Indoor Farmer Market",
        "distance_miles": 0.26,
        "coordinates": {
          "lat": 28.5417616,
          "lon": -81.3776219
        },
        "type": "supermarket"
      },
      {
        "name": "Jr Food Store",
        "distance_miles": 0.36,
        "coordinates": {
          "lat": 28.5385258,
          "lon": -81.3850895
        },
        "type": "convenience"
 